[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/14_kv_cache.ipynb)

# 🔴 Hard: KV Cache Attention

Реализуйте **multi-head attention с KV cache** для эффективной авторегрессионной генерации. Cache хранится отдельно для каждого слоя: прошлые hidden states больше не нужно снова проецировать в K и V на каждом шаге.

During LLM inference, recomputing all key/value projections at every step is wasteful.
A **KV cache** stores previously computed K and V tensors so only the new token(s) need projection. Q прошлых токенов не нужен: при decode нас интересует только output новой позиции.

## Prefill и decode

**Prefill:** модель получает весь prompt `(B,S,D)`, строит Q/K/V всех позиций и использует causal mask. Возвращённые K/V формы `(B,H,S,d_h)` становятся cache.

**Single-token decode:** вход имеет `(B,1,D)`. Новые K/V добавляются к cache по sequence-оси, запрос имеет только одну позицию и может смотреть на все сохранённые позиции, поэтому дополнительная causal mask не нужна. Длина cache после шага увеличивается на 1.

Например, prompt из 4 токенов создаёт cache длины 4. На шаге `t4` вычисляются только `Q4,K4,V4`, attention weights имеют форму `(B,H,1,5)`, а новый cache — `(B,H,5,d_h)`. При корректных одинаковых весах последовательный decode должен совпасть с соответствующей последней позицией полного causal forward.

> Уточнение границы: отсутствие маски корректно для требуемого **одного** нового токена. Если за раз декодировать несколько новых токенов, им понадобится causal mask со смещением на `S_past`.

### Signature
```python
class KVCacheAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x: torch.Tensor, cache=None) -> tuple[torch.Tensor, tuple]:
        # x: (B, S_new, D) — new tokens
        # cache: None or (K_past, V_past) each (B, num_heads, S_past, d_k)
        # Returns: (output, (K_all, V_all))
```

### Requirements
- Inherit from `nn.Module`
- `self.W_q`, `self.W_k`, `self.W_v`, `self.W_o`: `nn.Linear` projections
- When `cache=None` (prefill): apply **causal mask**, return all K/V as cache
- When `cache` provided (decode): concat new K/V with cached, no causal mask needed for single-token decode
- Incremental decode must produce **identical** results to full forward pass

### Key Idea
```
Prefill:  [t0 t1 t2 t3] → full causal attention → cache = (K_{0:3}, V_{0:3})
Decode:   [t4]           → Q=t4, K/V=cache+t4  → cache = (K_{0:4}, V_{0:4})
Decode:   [t5]           → Q=t5, K/V=cache+t5  → cache = (K_{0:5}, V_{0:5})
```

## Цена памяти

Cache растёт линейно с длиной контекста, batch, числом слоёв и K/V-голов: примерно `2 * layers * B * H_kv * S * d_h` элементов. Он ускоряет decode, но становится одним из главных потребителей памяти; поэтому GQA/MQA и cache quantization особенно полезны на длинных контекстах.

## План реализации

1. Спроецируйте только переданный `x` и разложите Q/K/V по головам.
2. Если cache есть, конкатенируйте прошлые и новые K/V строго по sequence dimension.
3. Получите scaled scores Q_new против K_all; маскируйте prefill, но не single-token decode.
4. Объедините головы, примените `W_o` и верните output вместе с обновлённым cache.

## Быстрые проверки

- Prefill cache содержит ровно `S` позиций, после decode — `S+1`.
- Старый префикс cache после concat остался неизменным.
- Decode одного токена совпадает с последней позицией полного causal расчёта на concatenated sequence.
- Batch, heads и head dimension не меняются при росте cache.

## Частые ошибки

- Конкатенировать по оси голов вместо sequence.
- Повторно проецировать cached K/V как hidden states.
- Наложить обычную `(1,1)` causal mask при decode и закрыть весь past.
- Сравнивать incremental output с full output при разных параметрах модулей.

## Материалы для глубокого изучения

- [Hugging Face: Cache explanation](https://huggingface.co/docs/transformers/cache_explanation) — формы cache и пошаговая генерация.
- [Hugging Face: KV cache strategies](https://huggingface.co/docs/transformers/kv_cache) — dynamic/static/offloaded/quantized cache.
- [PyTorch Transformer building blocks](https://docs.pytorch.org/tutorials/intermediate/transformer_building_blocks.html) — attention layouts и GQA как способ сократить K/V.

## Вопросы для самопроверки

1. Почему cache хранит K/V, но не обязан хранить Q?
2. Когда decode всё же требует causal mask?
3. От каких размеров линейно зависит память cache?

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class KVCacheAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        pass  # Initialize W_q, W_k, W_v, W_o

    def forward(self, x, cache=None):
        # 1. Project Q, K, V from x
        # 2. Reshape to multi-head: (B, num_heads, S, d_k)
        # 3. If cache exists, concat new K/V with cached K/V
        # 4. Compute attention (causal mask needed during prefill)
        # 5. Return (output, (K_all, V_all))
        pass

In [ ]:
# 🧪 Debug
torch.manual_seed(0)
attn = KVCacheAttention(d_model=64, num_heads=4)
x = torch.randn(1, 6, 64)

# Full forward
full_out, _ = attn(x)
print("Full output shape:", full_out.shape)  # (1, 6, 64)

# Incremental: prefill 4, decode 1, decode 1
out1, cache = attn(x[:, :4])
print("Cache K shape:", cache[0].shape)  # (1, 4, 4, 16)
out2, cache = attn(x[:, 4:5], cache=cache)
out3, cache = attn(x[:, 5:6], cache=cache)
inc_out = torch.cat([out1, out2, out3], dim=1)
print("Match:", torch.allclose(full_out, inc_out, atol=1e-5))

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('kv_cache')